# Formation Energies and Convex Hull (Analyzer)

Analyze existing total energy results to compute formation energies and build the convex hull.

<h2 style="color:green">Usage</h2>

1. Set the material set and formulas in section 1 below.
2. Click "Run" > "Run All".
3. The notebook finds materials, retrieves total energies, and builds the convex hull.

**Prerequisites:**
1. Save materials to a material set (for simpler search).
2. Relax if needed and store relaxed structures there.
3. Calculate total energies for the materials with the same formalism (e.g. DFT functional) and needed precision.

## How it works

1. **Find materials** in a material set (or all materials) matching the given formulas
2. **Preview** the materials (formula, structure type, space group)
3. **Retrieve total energies** from completed jobs
4. **Build convex hull** using pymatgen PhaseDiagram
5. **Analyze stability** — formation energies, energy above hull, decomposition products

## 1. Set up the environment and parameters

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages
await install_packages("made|api_examples|analyze_convex_hull")


### 1.2. Set parameters and configurations

In [ ]:
# Formulas to search for materials
FORMULAS = ["Hf", "Zr", "O2", "HfO2", "ZrO2"]

# Material set name to find materials and properties that belong to them. None = all materials.
MATERIAL_SET = None

# Property group — encodes application:model:subtype:functional (e.g. "qe:dft:gga:pbe").
GROUP = "qe:dft:gga:pbe"

# Precision value. None = use the best (highest) available. Specific value = use exactly that.
PRECISION = None

# Organization to select.
ORGANIZATION_NAME = None


## 2. Authenticate and initialize API client

### 2.2. Initialize API client

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()

selected_account = client.my_account
if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)
ACCOUNT_ID = selected_account.id
print(f"✅ Account: {selected_account.name} ({ACCOUNT_ID})")

## 3. Find materials

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import list_materials_by_set

set_materials = None
if MATERIAL_SET:
    set_materials = list_materials_by_set(client, ACCOUNT_ID, MATERIAL_SET)
    print(f"✅ Using set '{MATERIAL_SET}': {len(set_materials)} material(s)")
else:
    print("ℹ️  No material set specified — searching all materials in account.")


### 3.2. Search materials by formula

In [ ]:
# Search materials by formula (optionally restricted to MATERIAL_SET)
all_materials = []

for formula in FORMULAS:
    if set_materials is not None:
        matches = [m for m in set_materials if m.get("formula") == formula]
    else:
        matches = client.materials.list({"formula": formula, "owner._id": ACCOUNT_ID})
        matches = [m for m in matches if not m.get("isEntitySet")]
    for material in matches:
        material["_search_formula"] = formula  # track which formula query found this
    all_materials.extend(matches)

    print(f"{formula}: {len(matches)} material(s) found")

print(f"\nTotal: {len(all_materials)} materials")


### 3.3. Preview materials

In [ ]:
import pandas as pd

preview = []
for material in all_materials:
    n_atoms = len(material.get("basis", {}).get("coordinates", []))
    lattice_type = material.get("lattice", {}).get("type", "?")
    preview.append({
        "ID": material["_id"],
        "Name": material.get("name", "?"),
        "Formula": material.get("formula", "?"),
        "Lattice": lattice_type,
        "Atoms": n_atoms,
    })

df_preview = pd.DataFrame(preview)
df_preview

## 4. Retrieve total energies

For each material, find completed jobs and extract total energy properties.

In [ ]:
from collections import Counter

exabyte_ids = list(set(material["exabyteId"] for material in all_materials if material.get("exabyteId")))

# Query properties per exabyteId to avoid API pagination truncation
properties_by_exabyte_id = {}
for exabyte_id in exabyte_ids:
    query = {
        "exabyteId": exabyte_id,
        "data.name": "total_energy",
    }
    if GROUP:
        query["group"] = {"$regex": GROUP}
    if PRECISION is not None:
        query["precision.value"] = PRECISION
    props = client.properties.list(query=query)
    for property_holder in props:
        eid = property_holder.get("exabyteId")
        eid = eid if isinstance(eid, str) else (eid[0] if eid else None)
        if not eid:
            continue
        existing = properties_by_exabyte_id.get(eid)
        if existing is None or property_holder.get("precision", {}).get("value", 0) > existing.get("precision", {}).get("value", 0):
            properties_by_exabyte_id[eid] = property_holder

seen_exabyte_ids = set()
entries_data = []
for material in all_materials:
    exabyte_id = material.get("exabyteId")
    if not exabyte_id or exabyte_id in seen_exabyte_ids:
        continue
    seen_exabyte_ids.add(exabyte_id)

    property_holder = properties_by_exabyte_id.get(exabyte_id)
    if not property_holder:
        print(f"\u26a0\ufe0f  {material['formula']}: no total energy")
        continue

    elements = material["basis"]["elements"]
    number_of_atoms = len(material["basis"]["coordinates"])
    composition = dict(Counter(element["value"] for element in elements))
    energy = property_holder["data"]["value"]
    precision = property_holder.get("precision", {}).get("value", "?")

    entries_data.append({"material_id": material["_id"], "formula": material["formula"], "composition": composition, "n_atoms": number_of_atoms, "total_energy": energy})
    print(f"\u2705 {material['formula']} (precision={precision}): {energy:.4f} eV ({energy / number_of_atoms:.4f} eV/atom)")

print(f"\n{len(entries_data)} entries.")


## 5. Build convex hull

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from mat3ra.notebooks_utils.core.entity.property.analysis import (
    build_convex_hull, get_results_table
)
from mat3ra.notebooks_utils.ipython.entity.property.plot import (
    plot_convex_hull
)

phase_diagram = build_convex_hull(entries_data)
print(f"✅ Convex hull built with {len(entries_data)} entries.")


## 6. Results

In [ ]:
df_results = get_results_table(phase_diagram, entries_data)
df_results


## 7. Plot

In [ ]:
from mat3ra.notebooks_utils.ipython.plot._plotly import render_figure

fig = plot_convex_hull(phase_diagram)
render_figure(fig)
